## Recommendation Models

To run on GPU use this in terminal: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

### Vector Space Model

Term by Document Matrix (in this case Feature by Painting matrix)

Terms: feature dimensions (colour descriptors e.g. colour histogram, colour moment vector, colour palette, texture descriptors e.g. lbp, gabor filters, fractal dimenstions, wavelets, embeddings from cnns, gram matrix, etc...)

Document: painting (image)

Instead of TF-IDF, standardization + PCA are used where the value of that feature (after normalization, PCA, etc.) = term weight

Similarity: Use cosine/euclidian similarity to see which paintings are most similar to each other (to be used to for content-based)

It's used for information retreival (IR) to search for art with high

A painting is represented by one feature vector per feature type. E.g. Gram Matrix, colour histogram, wavelet feature, etc... Each of these is a feature vector, numerical representation of one visual characteristic. So, for painting P i:
v i(gram),v i(hist),v i(wavelet),v i(palette),v i(moment) 
are five separate feature vectors. Form the combined vector V i by concatinating/fusing each of the 5 five feature vectors with w j is the scaling/weighting coefficient to balance the influence of each feature type.

**Vector Space Model Recipe**

1. **Prepare Inventory and metadata**

Compute all the types of features (computed above) per image in the dataset and for each feature record the name, dimensionality, range, and how it was computed. Assign a stable canonical name and an ordering for each feature type. Example:

| Feature Type                    | Typical Size                                   | Example File / Key | Description                            |
| ------------------------------- | ---------------------------------------------- | ------------------ | -------------------------------------- |
| **Colour Moments Vector (CMV)** | 9 dims (3 channels × 3 stats: mean, std, skew) | `cmv_9d.npy`       | Simple color summary statistics        |
| **Wavelet Features**            | 32 dims                                        | `wavelet_32d.npy`  | Texture from multi-scale decomposition |
| **Colour Palette**              | 15 dims (5 colors × 3 channels)                | `palette_15d.npy`  | Dominant RGB colors                    |
| **Colour Histogram**            | 64 dims                                        | `hist_64d.npy`     | Global color distribution              |
| **Gram Matrix Features**        | 128 dims                                       | `gram_128d.npy`    | Style descriptor from CNN activations  |


Feature-Painting Matrix

| Feature ↓ / Painting →     | P₁    | P₂    | P₃    | P₄    | ... |
| -------------------------- | ----- | ----- | ----- | ----- | --- |
| CMV[0] (mean R)            | 0.12  | 0.10  | 0.15  | 0.11  | ... |
| CMV[1] (std G)             | 0.07  | 0.09  | 0.06  | 0.10  | ... |
| Wavelet[5]                 | -0.21 | -0.24 | -0.27 | -0.25 | ... |
| Palette[2] (dominant blue) | 0.45  | 0.12  | 0.60  | 0.10  | ... |
| Hist[14] (Hue bin)         | 0.08  | 0.01  | 0.09  | 0.03  | ... |
| Gram[87]                   | 0.10  | 0.22  | 0.16  | 0.19  | ... |
| ...                        | ...   | ...   | ...   | ...   | ... |


2. **Clean and Validate features**

Remove or Replace NaNs/Infinity/constant columns and ensure all feature vectors exist for each painting (so normalisation, PCA and similarity metrics won't work well). If some paintings have missing features impute/zero-fill/late-fusion. Look at distributions per feature type (plot them for fyp explanations). 

3. **Similarity metric and Normalisation strategy** 

Use Cosine Similarity or Euclidean; most commonly used cosine similarity since Euclidean is used for raw distances. For cosine use L2-normalisation per vector and for Euclidean use standardising per dimension. Perform cosine normalisation on the final combined feature vector (all feature vectors per feature type) per painting. Perform these steps at step 8.

4. **Preprocess each feature type separately**

Apply appropriate preprocessing based on the feature type e.g. for histograms use power-law normalisation and then L1/L2 normalise, for matrices flatten or use a specialised transformations (i.e. log diagonal, matrix) before normalisation, for raw numeric descriptors standardise (i.e. zero mean, unit variance) or scale to a consistent range, and for CNN embeddings center (i.e. subtract mean) and L2 normalise check re PCA/whitening. Imp. to treat diff feature types accordingly since it will bias the combined vector toward types with larger ranges. 

IF NEW PAINTINGS ARE ADDED AFTER DATASET: Save preprocessing parameters in pickled files e.g. scaler.pkl, pca.pkl, etc... to be loaded each time you need to compute features for a new image. The vector space model is only meaningful if every painting (old or new) is represented consistently in the same coordinate system, all feature vectors must lie in the same space to be compatible and meaningful. Pre-process it in exactly the same way as the training data.

5. **Dimensionality reduction and Decorrelation for each feature type**

Use PCA for high-dimensional descriptors like CNN layers, large histograms, etc... to reduce noise and dimension which will speed up nearest-neighbour search. Decide the target dimensions beforehand and find the balance between retained variance vs. storage/latency. Whitening components will make dimensions more istropic and avoid dominance of a few high-variance directions. Keep smaller, features (e.g. fractal dims, colour moments) as is if already low-dim.

6. **Weighting for each feature type** 

Start with uniform weights then plan to tune weights on a validation set using offline metrics to decide if certain features should be up/down weighted e.g. CNN embeddings/Gram matrix might be more powerful than raw texture descriptors. Save the weights to apply them consistently during concatenation. Imp. because raw concatenation without weighting gives disproportionate influence to features with higher dimensionality/variance (re previous step). 

7. **Fusion strategy**

Choose between early or late fusion. Early fusion will concatenate all the reduced feature vectors into one final vector per painting after pre-processing, PCA, and weighting steps. It's simple and has good performance for many tasks. Late fusion keeps seperate indices per feature and at query time, search each index and merge/score results like a weighted sum or learned combiner. This works for paintings with missing features and for implementing dynamic weighting so it's more felxible.

8. **Concatenate & finalize painting vectors**

Use the stable ordering of step 1, concatenate each painting's processed and weighted per-feature vector into one final vector per painting. After concatenation, L2-normalise each final vector (if cosine similarity was used in step 3). Store metadata that maps vector slices to the painting including feature positions (start/end indices) for each painting to be able to interpret the feature matrix. Meta data would include the following for each painting, stored in json or csv:
 {
  "P001": {
    "filename": "vangogh_starrynight.jpg",
    "artist": "Vincent van Gogh",
    "feature_indices": {
      "color_moments": [0, 9],
      "wavelets": [9, 29],
      "palette": [29, 44],
      "histogram": [44, 108],
      "gram_matrix": [108, 144]
    }
    "feature_weights": {
        "colour_moments": 0.3, 
        "wavelets": 0.2, 
        "palette": 0.3,
        "histogram": 0.4,
        "gram_matrix": 0.7
    }
  }
}

Final normalisation makes similarity computations consistent and enables cosine to reflect angular similarity across combined features. Hence, after normalisation perform cosine similarity.

9. **Build an efficient vector index** 

Choose an ANN (approx nearest neighbour) index; will probably use FAISS since dataset is relitively medium/large. ANN indexes enable fast top-k retrieval for real-time recommendations at scale. Decide on storage format, whether to use compression, and index update strategy (rebuild vs increment).

10. **Build user profiles**

The user profile is the query in the VSM, so the structure and type of profile determines personalsation quality. To create a user profile use both explicit and implicit data t. Explicit feedback includes user inputs such as name, age, gender, adding to favourites/not interested, add to gallary, survey/questionnaire (at the beginning before entering website). Implicit feedback includes time spent viewing the image, interaction history such as clicks, views, searches, analysing click streams, browsing duration, interaction with various item categories, social connections for collaborative filtering. The type of user profile is important for the filtering method to be used later on. Demographic profiles - based on age, gender, location etc... pros: useful for cold start when painting info is limited and cons: recommendations are less specific to the user's interest/lacks adaptivity, Content-based profiles - represent user interest by analysing features of paintings they've interacted with (user profile stored as a vector of keywords/terms) pros: effective for recommending niche items and provides high degree of transparency and cons: can lead to over-specialisation, recommending only items similar to what the user already likes, Collaborative profiles - uses users with similar tastes to recommend new things to the user instead of paintings similar to previously liked by the user - hence its based on the users interaction history (e.g., like/dislike) and is used to find "neighbors" with similar preferences pros: uncovers new unexpected interests for the user and requires no explicit metadata cons: cold-satrt problems and data sparsity as a large number of ratings are needed, Hybrid profiles - these profiles combine multiple approaches to leverage the strengths of each method and overcome individual weaknesses e.g. a system might use explicit ratings (collaborative) and item genres (content-based) to refine suggestions. Build a user vector with user's interaction history and normalise it to match the item vector normalisation. 

11. **Retreival and Scoring**

Retrieval includes finding candidate paintings based on similarity, where an input query vector is either a user/painting vector. Indexing e.g. FAISS is used to store painting vectors for fast retrieval and when query vector is fed into the index it's asking for top k most similar vectors based on cosine similarity and then the index returns candidate IDs of these paintings. Brute-force searches through all vectors and is computationally expensive (O(N) per query). On the other hand, ANN indexing reduces this to approximately O(log N) or even constant-time for large datasets and allows real-time recommendations, even when you have thousands of paintings.

Scoring ranks these fast approx. retrieved candidates and refines them into accurate rankings. After performing weighted cosine similarity return the top N which are the final recommendations. Feature fusion allows you to balance style, color, content, and composition aspects depending on your recommendation goal (e.g., stylistic similarity vs. thematic similarity).

weighted cosine similarity since feature fusion is being used: score(q,i)=j∑ ​wj ​cos(qj​,ij​) where wj is the weight of the feature type j.

12. **Re-ranking**

Re-ranking includes post retieval adjustments for continuous good recommendations. These include filtering already seen paintings, blending collaborative with content based, MMR (maximum marginal relevance) for diversity and relevance to control a lot of paintings having the same artist and same style, and bias correction for popular artists that dominate (apply penalty). All this is important because raw similarity ranking can lack diversity. 

13. **Explainability**

3 dots at the side of painting pop up will include why the painting was recommended, what features contributed and the top contributing features. Using slice mapping will allow per feature contributions to be computed and surface the top contributing features as an explanation for that recommendation. Instead of raw numerics include user friendly layout and explanations. 

| Type                           | Description                                                                            | Example message
| ------------------------------ | -------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------ |
| **Feature-based**              | Explain recommendations based on input features (color, texture, CNN embeddings, etc.) | “Recommended because it shares a similar color palette and texture with your selected painting.” |
| **Model-based**                | Use the model’s internal reasoning (attention, embeddings, gradients) to explain       | “High similarity in VGG19 feature space indicates shared brushstroke style.”                     |
| **User-based (Collaborative)** | Explain based on other users’ behaviors                                                | “Users who liked this painting also liked this one.” (Not applicable to your current system)     |
| **Post-hoc visual**            | Visualize which image regions contributed most                                         | Saliency or Grad-CAM visualization of regions driving similarity                                 |
| **Prototype-based**            | Show representative examples (clusters or style archetypes)                            | “This belongs to the same ‘Impressionist’ visual cluster as Monet’s works.”                      |


14. **Evaluation Metrics and Testing**

15. Edge Cases, Tuning, Re-ranking TO-DO





In [ ]:
# all necessary imports are listed below
import os
import re
import random
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pywt
import unicodedata

# libraries to generate a desc. - removed
#from transformers import BlipProcessor, BlipForConditionalGeneration

# Libraries for neural style transfer 
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from PIL import Image

import h5py
import csv
from torchvision.models import vgg19, VGG19_Weights
from sklearn.decomposition import IncrementalPCA, PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import normalize
import faiss
import joblib 
import torch.nn.functional as func
import nbimporter 
from Text_Descriptors import text_features
from Colour_Descriptors import extract_colour_features
from Texture_Descriptors import extract_lbp_features, extract_gabor_features, extract_wavelet_features, fractal_dimension
#from Local_Descriptors import detect_sift, detect_fast, detect_orb, visualize_keypoints

print(torch.__version__)

2.5.1+cu121


In [ ]:
dataset_path = r'C:\Users\danie\Desktop\_\Daniela Curmi\University\Third Year\Final Year Project\Final Year Project Code Implementation\Website\paintings'

In [ ]:
text_features(
    img_path=r'C:\Users\danie\Desktop\_\Daniela Curmi\University\Final Year Project\Final Year Project Code Implementation\Website\paintings\Impressionism\alfred-sisley_morning-in-june-saint-mammes-et-les-coteaux-de-la-celle-1884.jpg'
)

In [ ]:
# Add Text feature vector for painting-artist metadata 

#### Feature Extraction 

In [ ]:
# method used when corrupted filename crashed code snippet
# start off from last painting id
def get_last_saved_id(metadata_path):
    if not os.path.exists(metadata_path) or os.stat(metadata_path).st_size == 0:
        return -1  # nothing saved yet

    with open(metadata_path, "r", encoding="utf-8") as f:
        rows = f.readlines()

    if len(rows) <= 1:
        return -1  # only header exists

    last_row = rows[-1].strip().split(",")
    return int(last_row[0])

In [ ]:
# TO-FIX corrupted file names
bad_text = "waldmÔö£ð│Ôö¼ÔòØller"
replacement = "waldmuller"
count = 0

for root, dirs, files in os.walk(dataset_path):
    for filename in files:
        if bad_text in filename:
            
            old_path = os.path.join(root, filename)
            new_filename = filename.replace(bad_text, replacement)
            new_path = os.path.join(root, new_filename)
            
            try:
                os.rename(old_path, new_path)
                print(f"RENAMED:\n  {old_path}\n  to {new_path}\n")
                count += 1
            except Exception as e:
                print(f"FAILED TO RENAME:\n  {old_path}\n  Error: {e}\n")

print(f"Total files renamed: {count}")      

RENAMED:
  C:\Users\danie\Desktop\'\Daniela Curmi\University\Third Year\Final Year Project\Final Year Project Code Implementation\Website\paintings\Romanticism\ferdinand-georg-waldmÔö£ð│Ôö¼ÔòØller_view-of-the-dachstein-with-the-hallst-ttersee-from-the-h-tteneckalpe-at-ischl-1838.jpg
  → C:\Users\danie\Desktop\'\Daniela Curmi\University\Third Year\Final Year Project\Final Year Project Code Implementation\Website\paintings\Romanticism\ferdinand-georg-waldmuller_view-of-the-dachstein-with-the-hallst-ttersee-from-the-h-tteneckalpe-at-ischl-1838.jpg

Done! Total files renamed: 1


In [ ]:
features_path = "features_extracted.h5"
metadata_path = "metadata_features_extracted.csv"

# create HDF5 and CSV files to store extracted featres and metadata
if not os.path.exists(features_path):
    with h5py.File(features_path, "w") as f:
        pass

if not os.path.exists(metadata_path):
    with open(metadata_path, "w", newline='', encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["painting_id", "genre", "artist", "painting_name", "year"])

# function to save features for all paintings to HDF5
def save_features(painting_id, features_dict):
    with h5py.File(features_path, "a") as f:
        grp = f.require_group(str(painting_id))
        for name, vec in features_dict.items():
            if name in grp:
                del grp[name]
            grp.create_dataset(name, data=np.array(vec))

# extract Features for all the paintings in the dataset
def feature_extractor(dataset_path):
    vgg = vgg19(weights=VGG19_Weights.DEFAULT).features.eval()
    style_layers = ['0', '5', '10', '19', '28']
    
    # auto detect where to resume
    last_id = get_last_saved_id(metadata_path)
    painting_id = last_id + 1
    print(f"Resuming from painting_id = {painting_id}")
    
    current_index = 0
    failed = 0

    for root, dirs, files in os.walk(dataset_path):
        for filename in files: 
            if not filename.lower().endswith((".jpg", ".png", ".jpeg")):
                continue

            # skip until reach resume point (this was only used when fixing corrupted names)
            if current_index < painting_id:
                current_index += 1
                continue
            
            image_path = os.path.join(root, filename)

            data = text_features(image_path)
            if data is None:
                print("Failed to load image")
                failed += 1
                continue

            image_rgb, genre, artist, genre, painting_name, year = data
            
            moments, palette, _, _, hist_features, _ = extract_colour_features(image_rgb)

            painting_tensor = load_image(image_path)
            with torch.no_grad():
                gram_features = extract_gram_features(painting_tensor, vgg, style_layers)
            gram_vector = np.concatenate([g.flatten().cpu().numpy() for g in gram_features])

            with open(metadata_path, "a", newline='', encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([painting_id, genre, artist, painting_name, year])

            save_features(painting_id, {
                "colour_moments": moments,
                "colour_palette": palette,
                "colour_histogram": hist_features,
                "gram_matrix": gram_vector})
            
            painting_id += 1
            current_index += 1

#feature_extractor(dataset_path)

[INFO] Resuming from painting_id = 13804


#### Feature Preperation 

In [ ]:
# feature validation
def validate_features(features_path):
    issues = []
    total = 0

    with h5py.File(features_path, "r") as f:
        for painting_id in f.keys():
            grp = f[painting_id]

            for feature_vector in grp.keys():
                total += 1
                feature = grp[feature_vector][()]
                feature = np.array(feature, dtype=np.float64)

                # check for NaNs, infinities, and constant and empty columns
                has_nan = np.isnan(feature).any()
                has_infinity = np.isinf(feature).any()
                is_constant = (np.ptp(feature) == 0)
                is_empty = (feature.size == 0)

                if has_nan or has_infinity or is_constant or is_empty:
                    issues.append({
                        "painting_id": painting_id,
                        "feature": feature_vector,
                        "nan": has_nan,
                        "inf": has_infinity,
                        "constant": is_constant,
                        "empty": is_empty,
                        "size": feature.size,
                    })
    return issues, total

issues, total = validate_features("features_extracted.h5")

print(f"Total feature vectors checked: {total}")
print(f"Total issues detected: {len(issues)}\n")

# count issue types and output results
nan_count = sum(1 for x in issues if x["nan"])
inf_count = sum(1 for x in issues if x["inf"])
const_count = sum(1 for x in issues if x["constant"])
empty_count = sum(1 for x in issues if x["empty"])

print(f"NaN values: {nan_count}")
print(f"Infinite values: {inf_count}")
print(f"Constant vectors: {const_count}")
print(f"Empty vectors: {empty_count}")

for item in issues:  
    print(item)

Total feature vectors checked: 325776
Total issues detected: 2960

NaN values: 2960
Infinite values: 0
Constant vectors: 0
Empty vectors: 0
{'painting_id': '1000', 'feature': 'colour_moments', 'nan': True, 'inf': False, 'constant': False, 'empty': False, 'size': 9}
{'painting_id': '1001', 'feature': 'colour_moments', 'nan': True, 'inf': False, 'constant': False, 'empty': False, 'size': 9}
{'painting_id': '1002', 'feature': 'colour_moments', 'nan': True, 'inf': False, 'constant': False, 'empty': False, 'size': 9}
{'painting_id': '1003', 'feature': 'colour_moments', 'nan': True, 'inf': False, 'constant': False, 'empty': False, 'size': 9}
{'painting_id': '1004', 'feature': 'colour_moments', 'nan': True, 'inf': False, 'constant': False, 'empty': False, 'size': 9}
{'painting_id': '1006', 'feature': 'colour_moments', 'nan': True, 'inf': False, 'constant': False, 'empty': False, 'size': 9}
{'painting_id': '1009', 'feature': 'colour_moments', 'nan': True, 'inf': False, 'constant': False, 'empt

In [8]:
# feature cleaning to fix colour moments vector NaNs
def clean_features(features_path):
    with h5py.File(features_path, "a") as f:
        for painting_id in f.keys():
            grp = f[painting_id]

            # convert NaNs to 0 
            for feature_vector in grp.keys():
                feature = grp[feature_vector][()]
                feature = np.array(feature, dtype=np.float64)
                feature[np.isnan(feature)] = 0.0
                clean_feature = feature

                # replace features in place, important cause of file size
                grp[feature_vector][...] = clean_feature

clean_features(r"D:\features_extracted.h5")

In [11]:
# feature pre-processing per feature type
def preprocess_colour_histogram(hist):
    hist = np.array(hist, dtype=np.float64)
    
    # normalise so bins sum to 1 thus scale invarient
    s = hist.sum()
    if s == 0:
        return hist   
    return hist / s

def preprocess_colour_moments(moments):
    moments = np.array(moments, dtype=np.float64)

    # normalise mean + std
    moments[0:6] = moments[0:6] / 255.0

    # stabilize skewness
    moments[6:9] = np.tanh(moments[6:9])
    return moments

# def z_score_norm(feature_vector):
#     feature_vector = np.array(feature_vector, dtype=np.float64)

#     # standardise
#     scaler = StandardScaler()
#     mean = np.mean(feature_vector)
#     std = np.std(feature_vector)
#     if std == 0:
#         return np.zeros_like(feature_vector)
#     return (feature_vector - mean) / std

def preprocess_palette(palette):
    palette = np.array(palette, dtype=np.float64)

    # ensure reshape is safe
    if palette.size % 3 == 0:
        palette = palette.reshape(-1, 3)

    # normalise RGB channels to [0,1]
    palette = palette / 255.0
    return palette.flatten()

def preprocess_gram_matrix(flattened_gram_matrix):
    g = np.array(flattened_gram_matrix, dtype=np.float64)

    def extract_upper_tri(mat):
        return mat[np.triu_indices_from(mat)]

    final_upper = []
    idx = 0

    # style layers = ['0','5','10','19','28'] 
    channel_sizes = [64, 128, 256, 512, 512]

    # reconstruct square gram matrix to extract upper triangle since gram matrix was flattened 
    # then extract upper triangle per layer 
    for c in channel_sizes:
        size = c * c
        block = g[idx:idx+size]
        idx += size

        mat = block.reshape(c, c)
        upper = extract_upper_tri(mat)
        final_upper.append(upper)

    final = np.concatenate(final_upper)
    #final_output = z_score_norm(final)

    return final

# def preprocess_gram_matrix(gram_features):
#     stats = []

#     for g in gram_features:
#         g = g.cpu().numpy()

#         # mean activation correlation
#         stats.append(np.mean(g))

#         # variance
#         stats.append(np.var(g))

#         # energy (Frobenius norm)
#         stats.append(np.linalg.norm(g))

#     return np.array(stats)

def preprocess_features(features_path):
    with h5py.File(features_path, "a") as f:
        total = len(f.keys())
        print(f"Total Paintings: {total}")

        for idx, painting_id in enumerate(f.keys()):
            grp = f[painting_id]

            # skip if already processed
            if grp.attrs.get("preprocessed", False):
                continue

            if "colour_histogram" in grp:
                raw = grp["colour_histogram"][()]
                grp["colour_histogram"][...] = preprocess_colour_histogram(raw)
                grp.attrs["preprocessed"] = True

            if "colour_moments" in grp:
                raw = grp["colour_moments"][()]
                grp["colour_moments"][...] = preprocess_colour_moments(raw)
                grp.attrs["preprocessed"] = True

            if "colour_palette" in grp:
                raw = grp["colour_palette"][()]
                processed = preprocess_palette(raw)
                del grp["colour_palette"]
                grp.create_dataset("colour_palette", data=processed)
                grp.attrs["preprocessed"] = True

            if "gram_matrix" in grp:
                raw = grp["gram_matrix"][()]
                processed = preprocess_gram_matrix(raw)
                del grp["gram_matrix"]
                grp.create_dataset("gram_matrix", data=processed)

            if idx % 500 == 0:
                print(f"Processed {idx}/{total}")

preprocess_features(r"D:\features_extracted.h5")

Total Paintings: 81444
Processed 0/81444
Processed 500/81444
Processed 1000/81444
Processed 1500/81444
Processed 2000/81444
Processed 2500/81444
Processed 3000/81444
Processed 3500/81444
Processed 4000/81444
Processed 4500/81444
Processed 5000/81444
Processed 5500/81444
Processed 6000/81444
Processed 6500/81444
Processed 7000/81444
Processed 7500/81444
Processed 8000/81444
Processed 8500/81444
Processed 9000/81444
Processed 9500/81444
Processed 10000/81444
Processed 10500/81444
Processed 11000/81444
Processed 11500/81444
Processed 12000/81444
Processed 12500/81444
Processed 13000/81444
Processed 13500/81444
Processed 14000/81444
Processed 14500/81444
Processed 15000/81444
Processed 15500/81444
Processed 16000/81444
Processed 16500/81444
Processed 17000/81444
Processed 17500/81444
Processed 18000/81444
Processed 18500/81444
Processed 19000/81444
Processed 19500/81444
Processed 20000/81444
Processed 20500/81444
Processed 21000/81444
Processed 21500/81444
Processed 22000/81444
Processed 2

#### PCA, Whitening, L2-Normalisation, and Weighting

In [13]:
# choose dims per feature - needs more testing to fine-tune dimension size
PCA_DIMS = {
    "colour_histogram": 32, # "colour_moments": 9,   
    "colour_palette": 12,
    "gram_matrix": 64
}

# assign feature weights according to importance - needs more testing + learning weights
WEIGHTS = {
    "colour_histogram": 1.0,
    "colour_moments": 1.0,
    "colour_palette": 1.0,
    "gram_matrix": 1.0
}

# number of paintings per batch for incremental PCA
BATCH = 512  

def fit_pca_models(h5_in_path, pca_models_dir):
    # standardisation scalers before PCA
    scalers = {
        "colour_histogram": StandardScaler(),
        "colour_moments": StandardScaler(),
        "colour_palette": StandardScaler(),
        "gram_matrix": StandardScaler()
    }

    # fit PCA models for histogram, pallette, and gram matrix
    pca_hist = PCA(n_components=PCA_DIMS["colour_histogram"], whiten=True)
    pca_pal  = PCA(n_components=PCA_DIMS["colour_palette"], whiten=True)
    inc_pca_gram = IncrementalPCA(
        n_components=PCA_DIMS["gram_matrix"], 
        batch_size=BATCH,
        whiten=True
    )

    hist_list, moments_list, pal_list, gram_list = [], [], [], []

    with h5py.File(h5_in_path, "r") as fin:
        for k in fin.keys():
            grp = fin[k]

            hist_list.append(grp["colour_histogram"][()])
            moments_list.append(grp["colour_moments"][()])
            pal_list.append(grp["colour_palette"][()])
            gram_list.append(grp["gram_matrix"][()])

    # Convert to arrays
    hist_arr = np.vstack(hist_list)
    moments_arr = np.vstack(moments_list)
    pal_arr  = np.vstack(pal_list)
    gram_arr = np.vstack(gram_list)

    # Global Standardisation
    hist_scaled = scalers["colour_histogram"].fit_transform(hist_arr)
    scalers["colour_moments"].fit_transform(moments_arr)
    pal_scaled  = scalers["colour_palette"].fit_transform(pal_arr)
    gram_scaled = scalers["gram_matrix"].fit_transform(gram_arr)

    # PCA on histogram and palette
    pca_hist.fit(hist_scaled)
    pca_pal.fit(pal_scaled)

    # incremental PCA for gram
    for start in range(0, len(gram_scaled), BATCH):
        inc_pca_gram.partial_fit(gram_scaled[start:start+BATCH])

    # Save PCA models and scalars 
    joblib.dump(scalers["colour_histogram"], f"{pca_models_dir}/scaler_hist.joblib")
    joblib.dump(scalers["colour_moments"], f"{pca_models_dir}/scaler_moments.joblib")
    joblib.dump(scalers["colour_palette"],  f"{pca_models_dir}/scaler_palette.joblib")
    joblib.dump(scalers["gram_matrix"],     f"{pca_models_dir}/scaler_gram.joblib")

    joblib.dump(pca_hist, f"{pca_models_dir}/pca_colour_histogram.joblib")
    joblib.dump(pca_pal,  f"{pca_models_dir}/pca_colour_palette.joblib")
    joblib.dump(inc_pca_gram, f"{pca_models_dir}/inc_pca_gram_matrix.joblib")

    return True

fit_pca_models("features_extracted.h5", "pca_models")

SCALERS = {
    "colour_histogram": joblib.load("pca_models/scaler_hist.joblib"),
    "colour_palette": joblib.load("pca_models/scaler_palette.joblib"),
    "gram_matrix": joblib.load("pca_models/scaler_gram.joblib"),
    "colour_moments": joblib.load("pca_models/scaler_moments.joblib")
}

PCAS = {
    "colour_histogram": joblib.load("pca_models/pca_colour_histogram.joblib"),
    "colour_palette": joblib.load("pca_models/pca_colour_palette.joblib"),
    "gram_matrix": joblib.load("pca_models/inc_pca_gram_matrix.joblib")
}

def apply_pca(name, vec):
    if name == "colour_moments":
        scaler = SCALERS[name]
        return scaler.transform(vec.reshape(1, -1)).flatten()
    
    if name not in PCAS:
        return vec
    
    vec_scaled = SCALERS[name].transform(vec.reshape(1, -1))
    vec_pca = PCAS[name].transform(vec_scaled)

    return vec_pca.flatten()

def l2_normalise(feature_vector):
    feature_vector = np.array(feature_vector, dtype=np.float64)
    norm = np.linalg.norm(feature_vector)

    # prevent validation errors
    if norm == 0 or np.isnan(norm) or np.isinf(norm):
        return np.zeros_like(feature_vector)
    
    return feature_vector/norm

def scale_by_dimension(vec):
    d = len(vec)
    if d == 0:
        return vec
    
    return vec / np.sqrt(d)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'features_extracted.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

#### Feature Fusion and L2-Normalisation

In [ ]:
def feature_fusion(features):  
    # apply PCA dimensionality reduction per feature type
    h_pca = apply_pca("colour_histogram", features["colour_histogram"])
    m_pca = apply_pca("colour_moments", features["colour_moments"])  
    p_pca = apply_pca("colour_palette", features["colour_palette"])
    g_pca = apply_pca("gram_matrix", features["gram_matrix"])         

    # Scale-by-Dimension andd L2-normalise each feature type
    h_norm = l2_normalise(scale_by_dimension(h_pca))
    m_norm = l2_normalise(scale_by_dimension(m_pca))
    p_norm = l2_normalise(scale_by_dimension(p_pca))
    g_norm = l2_normalise(scale_by_dimension(g_pca))

    # apply weights
    h_w = WEIGHTS["colour_histogram"] * h_norm
    m_w = WEIGHTS["colour_moments"] * m_norm
    p_w = WEIGHTS["colour_palette"] * p_norm
    g_w = WEIGHTS["gram_matrix"] * g_norm

    # concatenate into one feature vector per painting, EARLY FUSION FOR NOW 
    # might have to use LATE FUSION, then use weighted cosine-similarity  
    combined = np.concatenate([h_w, m_w, p_w, g_w])

    # final L2-normalisation
    combined = l2_normalise(combined)

    return combined

# Calculate per feature contribution to infer feature importance 
def feature_contributions(q, i, feature_slices):
    contributions = {}

    for name, (start, end) in feature_slices.items():
        q_sub = q[start:end]
        i_sub = i[start:end]

        sim = np.dot(q_sub, i_sub)  
        contributions[name] = sim

    return contributions

def build_vector(features, use_features): 
    vectors = []

    # apply PCA dimensionality reduction per feature type
    h_pca = apply_pca("colour_histogram", features["colour_histogram"])
    m_pca = apply_pca("colour_moments", features["colour_moments"])  
    p_pca = apply_pca("colour_palette", features["colour_palette"])
    g_pca = apply_pca("gram_matrix", features["gram_matrix"])         

    # Scale-by-Dimension andd L2-normalise each feature type
    h_norm = scale_by_dimension(l2_normalise(h_pca))
    m_norm = scale_by_dimension(l2_normalise(m_pca))
    p_norm = scale_by_dimension(l2_normalise(p_pca))
    g_norm = scale_by_dimension(l2_normalise(g_pca))

    # apply weights
    h_w = WEIGHTS["colour_histogram"] * h_norm
    m_w = WEIGHTS["colour_moments"] * m_norm
    p_w = WEIGHTS["colour_palette"] * p_norm
    g_w = WEIGHTS["gram_matrix"] * g_norm

    if "histogram" in use_features:
        vectors.append(h_w)

    if "moments" in use_features:
        vectors.append(m_w)

    if "palette" in use_features:
        vectors.append(p_w)

    if "gram" in use_features:
        vectors.append(g_w)

    combined = np.concatenate(vectors)
    return l2_normalise(combined)
 

In [ ]:
features_concat_path = "features_concatenated.h5"

if not os.path.exists(features_concat_path):
    with h5py.File(features_concat_path, "w") as f:
        pass

with h5py.File(features_path, "r") as f_in, \
    h5py.File(features_concat_path, "w") as f_out:

    for painting_id in f_in.keys():
        grp = f_in[painting_id]

        features = {
            "colour_histogram": grp["colour_histogram"][()],
            "colour_moments": grp["colour_moments"][()],
            "colour_palette": grp["colour_palette"][()].flatten(),
            "gram_matrix": grp["gram_matrix"][()]
        }

        vector = feature_fusion(features)
        f_out.create_dataset(painting_id, data=vector)


#### Cosine-Similarity and Vector Indexing

In [ ]:
# If using early fusion each vector already has weights since using the concatenated vector, 
# if late fusion use seperate vector per feature and implement weighted cosine similarity 
def cosine_similarity(feature_vector1, feature_vector2):
    return func.cosine_similarity(feature_vector1.reshape(1,-1), feature_vector2.reshape(1,-1))[0][0] 

def build_faiss_index(h5_path):
    vectors = []
    ids = []

    with h5py.File(h5_path, "r") as f:
        for k in f.keys():
            vec = f[k][()]
            vectors.append(vec.astype(np.float32))
            ids.append(int(k))

    X = np.vstack(vectors)
    
    dim = X.shape[1]

    index = faiss.IndexFlatIP(dim)  
    index.add(X)

    return index, ids

#### Feature Weight Tuning using Grid Search

In [ ]:
contributions = feature_contributions()
WEIGHTS = normalize(contributions)

# Grid Search for weights using validation metric

#### Retrieval and Scoring

In [ ]:
def retrieval_scoring(index, ids, query_vec, top_k=10):
    query_vec = query_vec.astype(np.float32).reshape(1, -1)

    scores, indices = index.search(query_vec, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "painting_id": ids[idx],
            "score": float(scores[0][i])
        })

    return results

#### User Profile vector 

In [ ]:
# 

In [ ]:
index, ids = build_faiss_index("features_concatenated.h5")

# Test User Profile vector (query)
query_vec = feature_fusion 
results = retrieval_scoring(index, ids, query_vec, top_k=10)

#### Evaluation Metrics

- Precision@K
- Recall@K
- nDCG
- Diversity (intra-list distance)
- Novelty

### k-NN 

### BERT

### CLIP